In [1]:
# !pip install -q unsloth==2025.4.7 datasets==3.5.1

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "up_proj",
        "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state = 42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.4.6 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [4]:
model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [5]:
from datasets import load_dataset

dataset_name = "5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca"
SYS_INSTRUCT = "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."
raw_dataset = load_dataset(dataset_name, split="train")

def convert_to_chat_format(conversations):
    messages = [{"role": "system", "content": SYS_INSTRUCT}]
    for msg in conversations:
        role = "user" if msg["from"] == "human" else "assistant"
        messages.append({"role": role, "content": msg["value"]})
    return messages

def format_prompt(example):
    messages = convert_to_chat_format(example["conversations"])
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

dataset = raw_dataset.map(format_prompt, remove_columns=raw_dataset.column_names)
dataset = dataset.map(tokenize_function, batched=True)

In [6]:
from transformers import TrainingArguments
from trl.trainer.sft_trainer import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    save_total_limit=4,
    logging_steps=50,
    output_dir="./checkpoint/llama3-1b-multi-conversation",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    remove_unused_columns=True,
    max_steps=500,
    bf16=True,
)

trainer=SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,697 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
50,4.593900
100,3.699400
150,3.674600
200,3.740300
250,3.683600
300,3.712200
350,3.733300
400,3.651600
450,3.678300
500,3.667200


TrainOutput(global_step=500, training_loss=3.7834352416992187, metrics={'train_runtime': 1499.017, 'train_samples_per_second': 5.337, 'train_steps_per_second': 0.334, 'total_flos': 9.6772256956416e+16, 'train_loss': 3.7834352416992187, 'epoch': 0.6297229219143576})

In [7]:
# load model from ckpt dir then save to huggingface hub
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    "./checkpoint/llama3-1b-multi-conversation/checkpoint-500",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [8]:
type(model)

peft.peft_model.PeftModelForCausalLM

In [9]:
# model.push_to_hub_merged(
#     "thuanan/Llama-3.2-1B-Instruct-Chat-sft",
#     commit_message="Merge weights to push to hub",
# )

In [10]:
type(tokenizer)

transformers.tokenization_utils_fast.PreTrainedTokenizerFast

In [11]:
# tokenizer.push_to_hub(
#     "thuanan/Llama-3.2-1B-Instruct-Chat-sft",
#     commit_message="Push tokenizer to hub",
# )

In [12]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=128,
    temperature=1.0,
    do_sample=False,
    num_return_sequences=1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.3
)


prompt = [
    {
        "role": "system",
        "content": "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."
    },
    {
        "role": "user",
        "content": """Hãy chỉnh sửa câu này để ngắn gọn hơn mà không mất đi ý nghĩa: "Trận đấu là một thất bại nặng nề mặc dù thực tế là cả đội đã tập luyện trong nhiều tuần."""
    },
    {
        "role": "assistant",
        "content": """Nhiều tuần huấn luyện của đội đã dẫn đến một thất bại nặng nề."""
    },
    {
        "role": "user",
        "content": """Bạn có thể đề xuất một số chiến lược mà nhóm có thể sử dụng để cải thiện hiệu suất của họ trong trận đấu tiếp theo không?"""
    }
]

chat_text = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    tokenize=False
)

inputs = tokenizer(
    chat_text,
    return_tensors="pt"
).to("cuda:0")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        generation_config=generation_config,
    )
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "assistant\n" in output_text:
    answer = output_text.split("assistant\n")[-1].strip()
else:
    answer = output_text.strip()

print("Assistant reply:", answer)

Assistant reply: Chắc chắn! Dưới đây là một vài mẹo về cách các nhà tổ chức và cầu thủ chơi bóng đá tiến hành lại những gì chúng ta gọi là “tranh cãi” hoặc sự bất đồng giữa hai bên trên sân trước khi bắt đầu cuộc đua vào cuối ngày hôm nay: - Đảm bảo rằng tất cả mọi người đều hiểu rõ mục tiêu chung cũng như tầm quan trọng của việc đạt được nó. Điều này sẽ giúp ngăn chặn xung đột và thúc đẩy tinh thần đoàn kết. - Hãy cho phép đối phương bày tỏ cảm xúc của mình thông qua ngôn ngữ cơ thể hay văn bản. Bằng
